In [8]:
import torch
import torch.nn as nn
import sys

# Because folder name has a hyphen: FoMo-Meta
sys.path.insert(0, "/home/xding2/FoMo-0D-explore/FoMo-Meta/model")
from transformer import TransformerModel


class NoPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len):
        super().__init__()

    def forward(self, x):
        return x


def build_fomo_equivalent_model(
    criterion,
    num_features: int,
    emsize: int = 200,
    nhid: int = 200,
    nlayers: int = 6,
    nhead: int = 2,
    dropout: float = 0.0,
    seq_len: int = 10,
):
    # same n_out logic as FoMo
    if isinstance(criterion, nn.GaussianNLLLoss):
        n_out = 2
    elif isinstance(criterion, nn.CrossEntropyLoss):
        # safer than criterion.weight.shape[0] when weight=None
        n_out = 2 if criterion.weight is None else criterion.weight.shape[0]
    else:
        n_out = 1

    encoder = nn.Linear(num_features, emsize)
    y_encoder = nn.Linear(1, emsize)
    pos_encoder = NoPositionalEncoding(emsize, seq_len * 2)

    model = TransformerModel(
        encoder=encoder,
        nhead=nhead,
        ninp=emsize,
        nhid=nhid,
        nlayers=nlayers,
        dropout=dropout,
        style_encoder=None,
        y_encoder=y_encoder,
        input_normalization=False,
        pos_encoder=pos_encoder,
        decoder_dict={"standard": (None, n_out)},
        init_method=None,
        efficient_eval_masking=True,
        decoder_once_dict={},
        num_global_att_tokens=0,
        # required by current FoMo-Meta/model/transformer_layer.py
        model_para_dict={"num_R": 500, "last_layer_no_R": False},
    )
    model.criterion = criterion
    return model


# ---- Example usage ----
criterion = nn.CrossEntropyLoss()
model = build_fomo_equivalent_model(
    criterion=criterion,
    num_features=100,
    emsize=200,
    nhid=200,
    nlayers=2,
    nhead=2,
    dropout=0.0,
    seq_len=10,
)

print(model)

using vanilla + router
last_layer_no_R=False, is_final_layer=False
using vanilla + router
last_layer_no_R=False, is_final_layer=True
Initialized decoder "standard" with (None, 2), nout=2
++++++++++++++++++++ [CAUTIOUS] Zero-initialising attention output projections. Make sure this is intentional. ++++++++++++++++++++
++++++++++++++++++++ [CAUTIOUS] Zero-initialising attention output projections. Make sure this is intentional. ++++++++++++++++++++
TransformerModel(
  (transformer_encoder): TransformerEncoderDiffInit(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (router_att): RouterMultiHeadAttention(
          (att_compressor): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=200, out_features=200, bias=True)
          )
          (att_recover): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=200, out_features=200, bias=True)
          )
          (dropout): Dropout(p=0.2, inpla

In [9]:
for name, param in model.state_dict().items():
    print(name, param.shape)

transformer_encoder.layers.0.router_att.router torch.Size([500, 1, 200])
transformer_encoder.layers.0.router_att.att_compressor.in_proj_weight torch.Size([600, 200])
transformer_encoder.layers.0.router_att.att_compressor.in_proj_bias torch.Size([600])
transformer_encoder.layers.0.router_att.att_compressor.out_proj.weight torch.Size([200, 200])
transformer_encoder.layers.0.router_att.att_compressor.out_proj.bias torch.Size([200])
transformer_encoder.layers.0.router_att.att_recover.in_proj_weight torch.Size([600, 200])
transformer_encoder.layers.0.router_att.att_recover.in_proj_bias torch.Size([600])
transformer_encoder.layers.0.router_att.att_recover.out_proj.weight torch.Size([200, 200])
transformer_encoder.layers.0.router_att.att_recover.out_proj.bias torch.Size([200])
transformer_encoder.layers.0.router_att.ln1.weight torch.Size([200])
transformer_encoder.layers.0.router_att.ln1.bias torch.Size([200])
transformer_encoder.layers.0.self_attn.in_proj_weight torch.Size([600, 200])
transf

In [3]:
import torch
ckpt_val = torch.load('/home/xding2/FoMo-0D-explore/ckpt/meta_baseline(2layers)_context5000.feat100.R500.LTFalse.gen1tr1True.reuse100.E1500.step1000.bs2.lr0.0001.emb256.hdim512.nhead4.nlayer2.ndevice4.T00_20260315_172152/seed0/last.ckpt')['state_dict']

In [5]:
for name, value in ckpt_val.items():
    if hasattr(value, "shape"):
        print(name, value.shape)
    else:
        print(name, type(value))

criterion.weight torch.Size([2])
model.transformer_encoder.layers.0.router_att.router torch.Size([500, 1, 256])
model.transformer_encoder.layers.0.router_att.att_compressor.in_proj_weight torch.Size([768, 256])
model.transformer_encoder.layers.0.router_att.att_compressor.in_proj_bias torch.Size([768])
model.transformer_encoder.layers.0.router_att.att_compressor.out_proj.weight torch.Size([256, 256])
model.transformer_encoder.layers.0.router_att.att_compressor.out_proj.bias torch.Size([256])
model.transformer_encoder.layers.0.router_att.att_recover.in_proj_weight torch.Size([768, 256])
model.transformer_encoder.layers.0.router_att.att_recover.in_proj_bias torch.Size([768])
model.transformer_encoder.layers.0.router_att.att_recover.out_proj.weight torch.Size([256, 256])
model.transformer_encoder.layers.0.router_att.att_recover.out_proj.bias torch.Size([256])
model.transformer_encoder.layers.0.router_att.ln1.weight torch.Size([256])
model.transformer_encoder.layers.0.router_att.ln1.bias to